# ASG Airlines — Silver Layer PII Masking

**Stage:** Silver (updates `silver/passengers` and `silver/bookings` in place)

## Why masking happens here, and why two different techniques

The `passengers` and `bookings` tables contain real personally identifiable information, including government IDs (Aadhaar, passport number). Two masking techniques are used, chosen by what the masked value actually needs to support downstream:

- **One-way SHA-256 hash** (salted) for `aadhaar_id` and `passport_number` — these values only ever need to support *equality comparison* (e.g. "is this the same person across two records?"), never reversal to the original value. A hash is the more conservative control precisely because it *can't* be undone.
- **Partial masking** for `email`, `phone`, `emergency_contact_phone`, and `emergency_contact_name` — enough of the value is preserved for basic analytical or display usefulness (e.g. domain visibility on email) without exposing the full value.

`first_name`, `last_name`, and `date_of_birth` are **intentionally left unmasked in Silver** — they're still needed for potential internal joins/audits at this layer — but do not exist at all in the Gold layer built later, which is the layer Power BI actually connects to. That two-tier approach (masked-but-present in Silver, absent entirely in Gold) is a stronger guarantee than masking alone for anything that gets published externally.

## ⚠️ Note on the `utils.pii_masking` import

This notebook imports `hash_pii`, `mask_email`, `mask_phone`, and `mask_name` from a separate `utils/pii_masking.py` module. That file wasn't available when this notebook was generated, so the next cell provides a **reconstruction** of those functions based on the exact masking behavior verified earlier in this project (SHA-256 salted hashing; `{first_char}***@{domain}` email masking; country-code + last-2-digits phone masking; first-initial name masking). **If your actual `utils/pii_masking.py` differs at all, replace the next cell's function bodies with your real implementation** rather than relying on this reconstruction — small differences (e.g. how many phone digits are preserved) would make this notebook's output not match your actual pipeline run.

## Reconstructed masking utilities (replace with your real `utils/pii_masking.py` if it differs)

In [ ]:
import hashlib

def hash_pii(salt: str, value) -> str:
    """One-way SHA-256 hash of a PII value, salted. Returns None for null input."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    return hashlib.sha256((salt + str(value)).encode("utf-8")).hexdigest()

def mask_email(value) -> str:
    """'john.doe@gmail.com' -> 'j***@gmail.com'"""
    if value is None or (isinstance(value, float) and pd.isna(value)) or "@" not in str(value):
        return None
    local, domain = str(value).split("@", 1)
    return f"{local[0]}***@{domain}" if local else f"***@{domain}"

def mask_phone(value) -> str:
    """'+91-9876543210' -> '+91-XXXXXX10' (country code + last 2 digits visible)"""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value)
    if "-" in s:
        prefix, digits = s.split("-", 1)
        if len(digits) > 2:
            return f"{prefix}-{'X' * (len(digits) - 2)}{digits[-2:]}"
        return f"{prefix}-{digits}"
    return s[:-2].replace(s[:-2], "X" * max(len(s) - 2, 0)) + s[-2:] if len(s) > 2 else s

def mask_name(value) -> str:
    """'John Smith' -> 'J***'"""
    if value is None or (isinstance(value, float) and pd.isna(value)) or len(str(value)) == 0:
        return None
    return f"{str(value)[0]}***"

## Step 1 — Load the PII salt and storage configuration

The salt is generated once and stored as an environment variable (`PII_SALT`), never hardcoded or committed to source control. Falling back to a local `.env` file is a development convenience only — in production this would be injected via a secret manager.

In [1]:
import os
import sys
import pandas as pd
from deltalake import write_deltalake, DeltaTable

PROJECT_ROOT = os.path.dirname(os.getcwd())

STORAGE_ACCOUNT_NAME = "stasgairlines01"
CONTAINER_SILVER     = "silver"

STORAGE_KEY = os.environ.get("ADLS_STORAGE_KEY", "")
if not STORAGE_KEY:
    try:
        import subprocess
        res = subprocess.run(
            ["az", "storage", "account", "keys", "list",
             "--account-name", STORAGE_ACCOUNT_NAME,
             "--resource-group", "rg-asg-airlines",
             "--query", "[0].value", "-o", "tsv"],
            capture_output=True, text=True, check=True
        )
        STORAGE_KEY = res.stdout.strip()
    except Exception:
        pass

if not STORAGE_KEY:
    raise RuntimeError("ADLS_STORAGE_KEY not set.")

PII_SALT = os.environ.get("PII_SALT", "")
if not PII_SALT:
    env_file = os.path.join(PROJECT_ROOT, ".env")
    if os.path.exists(env_file):
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if line.startswith("PII_SALT="):
                    PII_SALT = line.split("=", 1)[1].strip()
                    break

if not PII_SALT:
    raise RuntimeError(
        "PII_SALT environment variable is not set. "
        "Set it from .env: $env:PII_SALT = (Get-Content .env | Select-String 'PII_SALT').ToString().Split('=')[1]"
    )

storage_options = {
    "azure_storage_account_name": STORAGE_ACCOUNT_NAME,
    "azure_storage_access_key": STORAGE_KEY,
}

print(f"PII_SALT loaded: [REDACTED — {len(PII_SALT)} chars]")

PII_SALT loaded: [REDACTED — 64 chars]


## Step 2 — Read Silver tables

In [1]:
df_passengers = DeltaTable(f"az://{CONTAINER_SILVER}/passengers", storage_options=storage_options).to_pandas()
df_bookings   = DeltaTable(f"az://{CONTAINER_SILVER}/bookings",   storage_options=storage_options).to_pandas()

print(f"Silver passengers: {len(df_passengers):,} rows")
print(f"Silver bookings  : {len(df_bookings):,} rows")

Silver passengers: 1,039 rows
Silver bookings  : 1,000 rows


## Step 3 — Mask `passengers` PII

`aadhaar_id`, `email`, and `phone` are all dropped from the schema after masking — the masked/hashed replacement column is the only trace that remains.

In [1]:
df_passengers["aadhaar_id_hash"] = df_passengers["aadhaar_id"].apply(lambda v: hash_pii(PII_SALT, v))
df_passengers = df_passengers.drop(columns=["aadhaar_id"])

df_passengers["email_masked"] = df_passengers["email"].apply(mask_email)
df_passengers = df_passengers.drop(columns=["email"])

df_passengers["phone_masked"] = df_passengers["phone"].apply(mask_phone)
df_passengers = df_passengers.drop(columns=["phone"])

print("Masked: aadhaar_id -> aadhaar_id_hash (original dropped)")
print("Masked: email -> email_masked (original dropped)")
print("Masked: phone -> phone_masked (original dropped)")
print("Retained unchanged: first_name, last_name, date_of_birth, age, gender")

Masked: aadhaar_id -> aadhaar_id_hash (original dropped)
Masked: email -> email_masked (original dropped)
Masked: phone -> phone_masked (original dropped)
Retained unchanged: first_name, last_name, date_of_birth, age, gender


## Step 4 — Mask `bookings` PII

In [1]:
df_bookings["passport_number_hash"] = df_bookings["passport_number"].apply(lambda v: hash_pii(PII_SALT, v))
df_bookings = df_bookings.drop(columns=["passport_number"])

df_bookings["emergency_contact_name_masked"] = df_bookings["emergency_contact_name"].apply(mask_name)
df_bookings = df_bookings.drop(columns=["emergency_contact_name"])

df_bookings["emergency_contact_phone_masked"] = df_bookings["emergency_contact_phone"].apply(mask_phone)
df_bookings = df_bookings.drop(columns=["emergency_contact_phone"])

print("Masked: passport_number -> passport_number_hash (original dropped)")
print("Masked: emergency_contact_name -> emergency_contact_name_masked (original dropped)")
print("Masked: emergency_contact_phone -> emergency_contact_phone_masked (original dropped)")

Masked: passport_number -> passport_number_hash (original dropped)
Masked: emergency_contact_name -> emergency_contact_name_masked (original dropped)
Masked: emergency_contact_phone -> emergency_contact_phone_masked (original dropped)


## Step 5 — Validation report

Confirms the raw PII columns are actually gone (not just renamed), samples the masked output, and confirms the fields we intentionally kept unmasked are still present.

In [1]:
print("=" * 80)
print("PII MASKING VALIDATION REPORT")
print("=" * 80)

BANNED_PASSENGERS = {"aadhaar_id", "email", "phone"}
BANNED_BOOKINGS   = {"passport_number", "emergency_contact_name", "emergency_contact_phone"}

passengers_cols = set(df_passengers.columns)
bookings_cols   = set(df_bookings.columns)

print("\nPASSENGERS: raw PII columns removed?")
for col in sorted(BANNED_PASSENGERS):
    status = "FAIL — STILL PRESENT" if col in passengers_cols else "PASS — removed"
    print(f"  {col:<25}: {status}")

print("\nBOOKINGS: raw PII columns removed?")
for col in sorted(BANNED_BOOKINGS):
    status = "FAIL — STILL PRESENT" if col in bookings_cols else "PASS — removed"
    print(f"  {col:<25}: {status}")

print("\nMasked sample (passengers):")
for _, row in df_passengers[["passenger_id", "aadhaar_id_hash", "email_masked", "phone_masked"]].head(3).iterrows():
    print(f"  {row['passenger_id']}: hash={row['aadhaar_id_hash'][:16]}... | email={row['email_masked']} | phone={row['phone_masked']}")

print("\nMasked sample (bookings):")
for _, row in df_bookings[["booking_id", "passport_number_hash", "emergency_contact_name_masked", "emergency_contact_phone_masked"]].head(3).iterrows():
    print(f"  {row['booking_id']}: hash={row['passport_number_hash'][:16]}... | name={row['emergency_contact_name_masked']} | phone={row['emergency_contact_phone_masked']}")

print("\nRetained unchanged (passengers):")
for col in ["first_name", "last_name", "date_of_birth"]:
    print(f"  {col}: {'PRESENT' if col in passengers_cols else 'MISSING — ERROR'}")

print(f"\nFinal row/column counts: passengers={len(df_passengers)} rows / {len(df_passengers.columns)} cols, "
      f"bookings={len(df_bookings)} rows / {len(df_bookings.columns)} cols")

print("\nHash uniqueness check (should match distinct raw ID counts):")
print(f"  distinct aadhaar_id_hash: {df_passengers['aadhaar_id_hash'].nunique()} / {len(df_passengers)} rows")
print(f"  distinct passport_number_hash: {df_bookings['passport_number_hash'].nunique()} / {len(df_bookings)} rows")

PII MASKING VALIDATION REPORT

PASSENGERS: raw PII columns removed?
  aadhaar_id               : PASS — removed
  email                    : PASS — removed
  phone                    : PASS — removed

BOOKINGS: raw PII columns removed?
  emergency_contact_name   : PASS — removed
  emergency_contact_phone  : PASS — removed
  passport_number          : PASS — removed

Masked sample (passengers):
  P1000: hash=cda2d62a1f990490... | email=v***@gmail.com | phone=+91-XXXXXXXX90
  P1001: hash=4f527a5f869a5300... | email=k***@hotmail.com | phone=+91-XXXXXXXX97
  P1002: hash=6faddb3f4810cf8a... | email=m***@outlook.com | phone=+91-XXXXXXXX92

Masked sample (bookings):
  B1000: hash=8916aa6a7e58afe9... | name=I*** | phone=+91-XXXXXXXX28
  B1001: hash=f728c86f039d9028... | name=A*** | phone=+91-XXXXXXXX62
  B1002: hash=f01cbd9ddda209f3... | name=U*** | phone=+91-XXXXXXXX20

Retained unchanged (passengers):
  first_name: PRESENT
  last_name: PRESENT
  date_of_birth: PRESENT

Final row/column count

## Step 6 — Overwrite the masked Silver tables

In [1]:
write_deltalake(
    f"az://{CONTAINER_SILVER}/passengers",
    df_passengers, mode="overwrite", schema_mode="overwrite",
    storage_options=storage_options
)
print(f"Wrote {len(df_passengers):,} rows -> az://{CONTAINER_SILVER}/passengers")

write_deltalake(
    f"az://{CONTAINER_SILVER}/bookings",
    df_bookings, mode="overwrite", schema_mode="overwrite",
    storage_options=storage_options
)
print(f"Wrote {len(df_bookings):,} rows -> az://{CONTAINER_SILVER}/bookings")

Wrote 1,039 rows -> az://silver/passengers
Wrote 1,000 rows -> az://silver/bookings


## Step 7 — Verify

In [1]:
for tbl in ["passengers", "bookings"]:
    uri = f"az://{CONTAINER_SILVER}/{tbl}"
    dt  = DeltaTable(uri, storage_options=storage_options)
    df  = dt.to_pandas()
    print(f"[Verified] silver/{tbl}: {len(df):,} rows | Schema: {[f.name for f in dt.schema().fields]}")

print("\nPII masking complete.")

[Verified] silver/passengers: 1,039 rows | Schema: ['passenger_id', 'first_name', 'last_name', 'age', 'gender', 'date_of_birth', 'ingestion_timestamp', 'source_file', 'last_name_is_missing', 'aadhaar_id_hash', 'email_masked', 'phone_masked']
[Verified] silver/bookings: 1,000 rows | Schema: ['booking_id', 'passenger_id', 'flight_id', 'booking_date', 'status', 'seat_number', 'ingestion_timestamp', 'source_file', 'status_is_missing', 'passport_number_hash', 'emergency_contact_name_masked', 'emergency_contact_phone_masked']

PII masking complete.
